# CARE-Fusion — Colab / Kaggle runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dtlong1979/care-fusion-sentiment-detection-vi/blob/main/notebooks/CARE_Fusion_Colab.ipynb)

Chạy thực nghiệm CARE-Fusion trên GPU. **Runtime → Change runtime type → GPU** (T4 free, hoặc A100/L4 nếu Colab Pro).

Thứ tự: clone → cài deps + Java → preprocess (A) → resources (B) → train (D) → evaluate (F–G).

In [ ]:
# 0) GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1) Clone repo (đổi sang token/SSH nếu repo còn private)
%cd /content
!git clone https://github.com/dtlong1979/care-fusion-sentiment-detection-vi.git
%cd care-fusion-sentiment-detection-vi

In [ ]:
# 2) Cài Java (cho VnCoreNLP) + Python deps
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip install -q -r requirements.txt
!pip install -q -e .   # make `care_fusion` importable (src layout)

In [ ]:
# 3) (Tuỳ chọn) Mount Drive để lưu checkpoint qua các session
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/care-fusion', exist_ok=True)

In [ ]:
# 4) Part A — tiền xử lý (tải VnCoreNLP lần đầu)
!python -m care_fusion.preprocess --config configs/default.yaml

In [ ]:
# 5) Part B — q_j + PMI graph (chỉ từ train)
!python -m care_fusion.resources --config configs/default.yaml --steps q,pmi

## Phase 2/3 (sẽ bổ sung)

Các cell dưới sẽ được thêm khi `train.py` / `baselines.py` / `evaluate.py` hoàn thiện:

```bash
# baseline text-only + sinh OOF p_text -> weak labels (B2)
python -m care_fusion.baselines --config configs/default.yaml --model B1 --emit-oof
python -m care_fusion.resources --config configs/default.yaml --steps weak --ptext artifacts/p_text_oof.json

# huấn luyện CARE-Fusion (5 seed) + baselines + ablation
python -m care_fusion.train --config configs/default.yaml --out /content/drive/MyDrive/care-fusion

# đánh giá F1–F5 + kiểm định G
python -m care_fusion.evaluate --config configs/default.yaml --ckpt-dir /content/drive/MyDrive/care-fusion
```